In [23]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

#### Start by performing a GET request on the url above and convert the response into a BeautifulSoup object.

In [24]:
url = "https://realpython.github.io/fake-jobs/"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

print(response.status_code)

200


#### a. Use the .find method to find the tag containing the first job title ("Senior Python Developer"). Hint: can you find a tag type and/or a class that could be helpful for extracting this information? Extract the text from this title.

In [25]:
first_title = soup.find("h2", class_="title")
print(first_title.text.strip())

Senior Python Developer


#### b. Now, use what you did for the first title, but extract the job title for all jobs on this page. Store the results in a list.


In [26]:
titles = [h2.text.strip() for h2 in soup.find_all("h2", class_="title")]
print(titles)

['Senior Python Developer', 'Energy engineer', 'Legal executive', 'Fitness centre manager', 'Product manager', 'Medical technical officer', 'Physiological scientist', 'Textile designer', 'Television floor manager', 'Waste management officer', 'Software Engineer (Python)', 'Interpreter', 'Architect', 'Meteorologist', 'Audiological scientist', 'English as a second language teacher', 'Surgeon', 'Equities trader', 'Newspaper journalist', 'Materials engineer', 'Python Programmer (Entry-Level)', 'Product/process development scientist', 'Scientist, research (maths)', 'Ecologist', 'Materials engineer', 'Historic buildings inspector/conservation officer', 'Data scientist', 'Psychiatrist', 'Structural engineer', 'Immigration officer', 'Python Programmer (Entry-Level)', 'Neurosurgeon', 'Broadcast engineer', 'Make', 'Nurse, adult', 'Air broker', 'Editor, film/video', 'Production assistant, radio', 'Engineer, communications', 'Sales executive', 'Software Developer (Python)', 'Futures trader', 'Tour

#### c. Finally, extract the companies, locations, and posting dates for each job. For example, the first job has a company of "Payne, Roberts and Davis", a location of "Stewartbury, AA", and a posting date of "2021-04-08". Ensure that the text that you extract is clean, meaning no extra spaces or other characters at the beginning or end.


In [27]:
companies = [h3.text.strip() for h3 in soup.find_all("h3", class_="company")]
locations = [p.text.strip() for p in soup.find_all("p", class_="location")]
dates = [time["datetime"] for time in soup.find_all("time")]

print(companies[:3])
print(locations[:3])
print(dates[:3])

['Payne, Roberts and Davis', 'Vasquez-Davidson', 'Jackson, Chambers and Levy']
['Stewartbury, AA', 'Christopherville, AA', 'Port Ericaburgh, AA']
['2021-04-08', '2021-04-08', '2021-04-08']


#### d. Take the lists that you have created and combine them into a pandas DataFrame.

In [28]:
df = pd.DataFrame({
    "title": titles,
    "company": companies,
    "location": locations,
    "date": dates
})

df

,title,company,location,date
0,Senior Python Developer,"Payne, Roberts and Davis","Stewartbury, AA",2021-04-08
1,Energy engineer,Vasquez-Davidson,"Christopherville, AA",2021-04-08
2,Legal executive,"Jackson, Chambers and Levy","Port Ericaburgh, AA",2021-04-08
3,Fitness centre manager,Savage-Bradley,"East Seanview, AP",2021-04-08
4,Product manager,Ramirez Inc,"North Jamieview, AP",2021-04-08
...,...,...,...,...
95,Museum/gallery exhibitions officer,"Nguyen, Yoder and Petty","Lake Abigail, AE",2021-04-08
96,"Radiographer, diagnostic",Holder LLC,"Jacobshire, AP",2021-04-08
97,Database administrator,Yates-Ferguson,"Port Susan, AE",2021-04-08
98,Furniture designer,Ortega-Lawrence,"North Tiffany, AA",2021-04-08


#### 2. Next, add a column that contains the url for the "Apply" button. Try this in two ways.

In [29]:
job_cards = soup.find_all("div", class_="card-content")

apply_urls_bs = []
for card in job_cards:
    links = card.find_all("a")
    apply_urls_bs.append(links[-1]["href"])

print(apply_urls_bs[:3])

['https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html', 'https://realpython.github.io/fake-jobs/jobs/energy-engineer-1.html', 'https://realpython.github.io/fake-jobs/jobs/legal-executive-2.html']


#### a. First, use the BeautifulSoup find_all method to extract the urls.

In [30]:
import re

def title_to_slug(title):
    slug = title.lower()                        # lowercase everything
    slug = re.sub(r"[^a-z0-9\s-]", "", slug)   # remove special characters
    slug = re.sub(r"\s+", "-", slug.strip())    # spaces → hyphens
    slug = re.sub(r"-+", "-", slug)             # collapse double hyphens
    return slug

apply_urls_built = [
    f"https://realpython.github.io/fake-jobs/jobs/{title_to_slug(title)}-{i}.html"
    for i, title in enumerate(titles)
]

print(apply_urls_built[:3])

['https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html', 'https://realpython.github.io/fake-jobs/jobs/energy-engineer-1.html', 'https://realpython.github.io/fake-jobs/jobs/legal-executive-2.html']


#### b. Next, get those same urls in a different way. Examine the urls and see if you can spot the pattern of how they are constructed. Then, build the url using the elements you have already extracted. Ensure that the urls that you created match those that you extracted using BeautifulSoup. Warning: You will need to do some string cleaning and prep in constructing the urls this way. For example, look carefully at the urls for the "Software Engineer (Python)" job and the "Scientist, research (maths)" job.

In [31]:
mismatches = [(i, apply_urls_bs[i], apply_urls_built[i]) 
              for i in range(len(titles)) 
              if apply_urls_bs[i] != apply_urls_built[i]]

if mismatches:
    for i, bs_url, built_url in mismatches:
        print(f"Row {i}:")
        print(f"  BS:    {bs_url}")
        print(f"  Built: {built_url}")
else:
    print("All 100 URLs match!")

Row 21:
  BS:    https://realpython.github.io/fake-jobs/jobs/product-process-development-scientist-21.html
  Built: https://realpython.github.io/fake-jobs/jobs/productprocess-development-scientist-21.html
Row 25:
  BS:    https://realpython.github.io/fake-jobs/jobs/historic-buildings-inspector-conservation-officer-25.html
  Built: https://realpython.github.io/fake-jobs/jobs/historic-buildings-inspectorconservation-officer-25.html
Row 36:
  BS:    https://realpython.github.io/fake-jobs/jobs/editor-film-video-36.html
  Built: https://realpython.github.io/fake-jobs/jobs/editor-filmvideo-36.html
Row 52:
  BS:    https://realpython.github.io/fake-jobs/jobs/producer-television-film-video-52.html
  Built: https://realpython.github.io/fake-jobs/jobs/producer-televisionfilmvideo-52.html
Row 68:
  BS:    https://realpython.github.io/fake-jobs/jobs/designer-fashion-clothing-68.html
  Built: https://realpython.github.io/fake-jobs/jobs/designer-fashionclothing-68.html
Row 71:
  BS:    https://realp

In [32]:
def title_to_slug(title):
    slug = title.lower()
    slug = re.sub(r"[/,]", " ", slug)          # replace / and , with a space
    slug = re.sub(r"[^a-z0-9\s-]", "", slug)   # remove remaining special characters
    slug = re.sub(r"\s+", "-", slug.strip())    # spaces → hyphens
    slug = re.sub(r"-+", "-", slug)             # collapse double hyphens
    return slug

apply_urls_built = [
    f"https://realpython.github.io/fake-jobs/jobs/{title_to_slug(title)}-{i}.html"
    for i, title in enumerate(titles)
]

print(apply_urls_built[:3])

['https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html', 'https://realpython.github.io/fake-jobs/jobs/energy-engineer-1.html', 'https://realpython.github.io/fake-jobs/jobs/legal-executive-2.html']


In [33]:
mismatches = [(i, apply_urls_bs[i], apply_urls_built[i]) 
              for i in range(len(titles)) 
              if apply_urls_bs[i] != apply_urls_built[i]]

if mismatches:
    for i, bs_url, built_url in mismatches:
        print(f"Row {i}:")
        print(f"  BS:    {bs_url}")
        print(f"  Built: {built_url}")
else:
    print("All 100 URLs match!")

All 100 URLs match!


In [34]:
df["apply_url"] = apply_urls_bs

df

,title,company,location,date,apply_url
0,Senior Python Developer,"Payne, Roberts and Davis","Stewartbury, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/se...
1,Energy engineer,Vasquez-Davidson,"Christopherville, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/en...
2,Legal executive,"Jackson, Chambers and Levy","Port Ericaburgh, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/le...
3,Fitness centre manager,Savage-Bradley,"East Seanview, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/fi...
4,Product manager,Ramirez Inc,"North Jamieview, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/pr...
...,...,...,...,...,...
95,Museum/gallery exhibitions officer,"Nguyen, Yoder and Petty","Lake Abigail, AE",2021-04-08,https://realpython.github.io/fake-jobs/jobs/mu...
96,"Radiographer, diagnostic",Holder LLC,"Jacobshire, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/ra...
97,Database administrator,Yates-Ferguson,"Port Susan, AE",2021-04-08,https://realpython.github.io/fake-jobs/jobs/da...
98,Furniture designer,Ortega-Lawrence,"North Tiffany, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/fu...


#### Finally, we want to get the job description text for each job.

first_job_url = "https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html"

first_job_resp = requests.get(first_job_url)
first_job_soup = BeautifulSoup(first_job_resp.text, "html.parser")

description = first_job_soup.find("div", class_="content").find("p").text.strip()
print(description)

#### a. Start by looking at the page for the first job, https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html. Using BeautifulSoup, extract the job description paragraph.

In [35]:
def get_job_description(url):
    resp = requests.get(url)
    page_soup = BeautifulSoup(resp.text, "html.parser")
    return page_soup.find("div", class_="content").find("p").text.strip()

# Test it with the example from the assignment
test_url = "https://realpython.github.io/fake-jobs/jobs/television-floor-manager-8.html"
get_job_description(test_url)

'At be than always different American address. Former claim chance prevent why measure too. Almost before some military outside baby interview. Face top individual win suddenly. Parent do ten after those scientist. Medical effort assume teacher wall. Significant his himself clearly very. Expert stop area along individual. Three own bank recognize special good along.'

In [36]:
#### c. Use the .apply method on the url column you created above to retrieve the description text for all of the jobs.

In [37]:
df["description"] = df["apply_url"].apply(get_job_description)

df

,title,company,location,date,apply_url,description
0,Senior Python Developer,"Payne, Roberts and Davis","Stewartbury, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/se...,Professional asset web application environment...
1,Energy engineer,Vasquez-Davidson,"Christopherville, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/en...,Party prevent live. Quickly candidate change a...
2,Legal executive,"Jackson, Chambers and Levy","Port Ericaburgh, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/le...,Administration even relate head color. Staff b...
3,Fitness centre manager,Savage-Bradley,"East Seanview, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/fi...,Tv program actually race tonight themselves tr...
4,Product manager,Ramirez Inc,"North Jamieview, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/pr...,Traditional page a although for study anyone. ...
...,...,...,...,...,...,...
95,Museum/gallery exhibitions officer,"Nguyen, Yoder and Petty","Lake Abigail, AE",2021-04-08,https://realpython.github.io/fake-jobs/jobs/mu...,Paper age physical current note. There reality...
96,"Radiographer, diagnostic",Holder LLC,"Jacobshire, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/ra...,Able such right culture. Wrong pick structure ...
97,Database administrator,Yates-Ferguson,"Port Susan, AE",2021-04-08,https://realpython.github.io/fake-jobs/jobs/da...,Create day party decade high clear. Past trade...
98,Furniture designer,Ortega-Lawrence,"North Tiffany, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/fu...,Pressure under rock next week. Recognize so re...


## BONUS QUESTIONS

In [38]:
billboard_url = "https://www.billboard.com/charts/hot-100/"
headers = {"User-Agent": "Mozilla/5.0"}

bb_response = requests.get(billboard_url, headers=headers)
print(bb_response.status_code)

200


In [40]:
bb_soup = BeautifulSoup(bb_response.text, "html.parser")

# Let's find the #1 song first
first_song = bb_soup.find("div", class_="o-chart-results-list-row-container")
print(first_song)


<div class="o-chart-results-list-row-container">
<ul class="o-chart-results-list-row // lrv-a-unstyle-list lrv-u-flex lrv-u-background-color-white a-chart-has-chart-detail u-border-radius-a-6 u-border-radius-a-0@mobile-max lrv-u-margin-b-1 u-margin-b-0.625@mobile-max lrv-u-padding-tb-050 lrv-u-align-items-center lrv-u-padding-r-050@mobile-max lrv" data-ajax="" data-detail-target="1">
<li class="o-chart-results-list__item // lrv-u-color-black u-width-60 u-width-45px@mobile-max lrv-u-flex lrv-u-flex-direction-column lrv-u-flex-shrink-0 lrv-u-align-items-center lrv-u-justify-content-center u-height-60 lrv-u-background-color-brand-primary@desktop">
<span class="c-label a-font-basic u-font-size-33@desktop u-font-size-19@mobile-max u-line-height-33px">
	
	1
</span>
<div class="c-svg u-height-10@mobile-max u-width-10@mobile-max u-hidden@tablet">
<svg viewbox="0 0 417.06 504.22" xmlns="http://www.w3.org/2000/svg"><path d="m178.78 13.48-.04-.04L9 183.17l.09.09C3.45 189.75 0 198.22 0 207.5c0 20.

In [41]:
# Get the #1 song title
song = first_song.find("h3", id="title-of-a-story").text.strip()

# Get the #1 artist
artist = first_song.find("span", class_="c-label a-no-trucate").text.strip()

print("Song:", song)
print("Artist:", artist)


AttributeError: 'NoneType' object has no attribute 'text'

In [42]:
# Print just the song title to confirm that works
print("Song:", song)

# Look at all the span tags inside the first song to find the artist
for span in first_song.find_all("span"):
    text = span.text.strip()
    if text:  # only print spans that have text
        print(repr(text))

Song: Choosin' Texas
'1'
'Ella Langley'
'LW'
'2'
'PEAK'
'1'
'WEEKS'
'24'
'LW'
'2'
'PEAK'
'1'
'WEEKS'
'24'
'39'
'11/01/25'
'1'
'02/14/26'
'Share Chart on Facebook'
'Facebook'
'Share Chart on Twitter'
'Twitter'
'Share Chart on Copy Link'
'Copy Link'
'RIAA Certification:'
'Platinum'


In [43]:
# Find names for data
for span in first_song.find_all("span"):
    text = span.text.strip()
    if text:
        print(repr(text), "→", span.get("class"))

'1' → ['c-label', 'a-font-basic', 'u-font-size-33@desktop', 'u-font-size-19@mobile-max', 'u-line-height-33px']
'Ella Langley' → ['c-label', 'a-no-trucate', 'a-font-secondary', 'u-font-size-15', 'u-font-size-13@mobile-max', 'u-line-height-18px@mobile-max', 'u-letter-spacing-0010', 'u-line-height-21px', 'a-children-link-color-black', 'a-children-link-color-brand-secondary:hover', 'lrv-a-children-link-decoration-underline:hover', 'lrv-u-display-block', 'a-truncate-ellipsis-2line', 'u-max-width-397', 'u-max-width-230@tablet-only', 'u-max-width-300@mobile-max']
'LW' → ['c-span', 'a-font-secondary', 'u-color-grey-141', 'lrv-u-text-align-right@desktop', 'lrv-u-text-align-left', 'u-font-size-12', 'u-line-height-normal', 'u-width-40', 'u-width-auto@mobile-max', 'u-min-width-auto@mobile-max', 'lrv-u-margin-r-025@mobile-max']
'2' → ['c-label', 'u-font-family-secondary@mobile-max', 'u-font-family-basic@tablet', 'u-font-weight-800@tablet', 'u-font-size-12', 'u-line-height-normal', 'lrv-u-padding-tb

In [44]:
# Extract the stat values (LW, Peak, Weeks) by finding all the value spans
stats = first_song.find_all("span", class_="c-label u-font-family-secondary@mobile-max u-font-family-basic@tablet u-font-weight-800@tablet u-font-size-12 u-line-height-normal lrv-u-padding-tb-00@mobile-max u-min-width-30px u-width-auto@mobile-max u-min-width-auto@mobile-max lrv-u-margin-r-050@mobile-max")

# There are duplicates so just take the first 3
last_week = stats[0].text.strip()
peak = stats[1].text.strip()
weeks = stats[2].text.strip()

print("Last Week:", last_week)
print("Peak:", peak)
print("Weeks:", weeks)

Last Week: 2
Peak: 1
Weeks: 24


In [45]:
def parse_card(card):
    # Song title
    song = card.find("h3", id="title-of-a-story").text.strip()
    
    # Artist
    artist = card.find("span", class_="a-no-trucate").text.strip()
    
    # This week rank
    rank = card.find("span", class_="u-font-size-33@desktop").text.strip()
    
    # Last week, peak, weeks — grab all stat values and take first 3
    stats = card.find_all("span", class_="c-label u-font-family-secondary@mobile-max u-font-family-basic@tablet u-font-weight-800@tablet u-font-size-12 u-line-height-normal lrv-u-padding-tb-00@mobile-max u-min-width-30px u-width-auto@mobile-max u-min-width-auto@mobile-max lrv-u-margin-r-050@mobile-max")
    last_week = stats[0].text.strip()
    peak = stats[1].text.strip()
    weeks = stats[2].text.strip()
    
    return {
        "this_week": rank,
        "song": song,
        "artist": artist,
        "last_week": last_week,
        "peak": peak,
        "weeks_on_chart": weeks
    }

# Test on the first song
parse_card(first_song)



{'this_week': '1',
 'song': "Choosin' Texas",
 'artist': 'Ella Langley',
 'last_week': '2',
 'peak': '1',
 'weeks_on_chart': '24'}

In [46]:
# ALL 100

all_cards = bb_soup.find_all("div", class_="o-chart-results-list-row-container")

records = []
for card in all_cards:
    try:
        records.append(parse_card(card))
    except Exception as e:
        print(f"Skipping a card due to error: {e}")

bb_df = pd.DataFrame(records)
print(f"Got {len(bb_df)} songs")
bb_df

Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute 'text'
Skipping a card due to error: 'NoneType' object has no attribute

,this_week,song,artist,last_week,peak,weeks_on_chart
0,1,Choosin' Texas,Ella Langley,2,1,24


In [47]:
# Look at the second card
second_card = all_cards[1]

for span in second_card.find_all("span"):
    text = span.text.strip()
    if text:
        print(repr(text), "→", span.get("class"))

'2' → ['c-label', 'a-font-basic', 'u-font-size-22@desktop', 'u-font-size-19@mobile-max', 'u-line-height-33px']
'BTS' → ['c-label', 'a-no-trucate', 'a-font-secondary', 'u-font-size-15', 'u-font-size-13@mobile-max', 'u-line-height-18px@mobile-max', 'u-letter-spacing-0010', 'u-line-height-21px', 'a-children-link-color-black', 'a-children-link-color-brand-secondary:hover', 'lrv-a-children-link-decoration-underline:hover', 'lrv-u-display-block', 'a-truncate-ellipsis-2line', 'u-max-width-397', 'u-max-width-230@tablet-only', 'u-max-width-300@mobile-max']
'LW' → ['c-span', 'a-font-secondary', 'u-color-grey-141', 'lrv-u-text-align-right@desktop', 'lrv-u-text-align-left', 'u-font-size-12', 'u-line-height-normal', 'u-width-40', 'u-width-auto@mobile-max', 'u-min-width-auto@mobile-max', 'lrv-u-margin-r-025@mobile-max']
'1' → ['c-label', 'u-font-family-secondary@mobile-max', 'u-font-family-basic@tablet', 'u-font-weight-800@tablet', 'u-font-size-12', 'u-line-height-normal', 'lrv-u-padding-tb-00@mobil

In [48]:
print(second_card.find("h3"))


<h3 class="c-title a-font-basic u-letter-spacing-0010 u-max-width-397 lrv-u-font-size-16 lrv-u-font-size-14@mobile-max u-line-height-22px u-word-spacing-0063 u-line-height-normal@mobile-max a-truncate-ellipsis-2line lrv-u-margin-b-025 lrv-u-margin-b-00@mobile-max" id="title-of-a-story">

	
	
		
					Swim		
	
</h3>


In [49]:
print(second_card.find("h3").text.strip())


Swim


In [50]:
def parse_card(card):
    # Song title - works for all cards
    song = card.find("h3", id="title-of-a-story").text.strip()
    
    # Artist - works for all cards
    artist = card.find("span", class_="a-no-trucate").text.strip()
    
    # Rank - #1 has different font size class than the rest
    rank = card.find("span", class_="u-font-size-33@desktop")
    if rank is None:
        rank = card.find("span", class_="u-font-size-22@desktop")
    rank = rank.text.strip()
    
    # Last week, peak, weeks
    stats = card.find_all("span", class_="c-label u-font-family-secondary@mobile-max u-font-family-basic@tablet u-font-weight-800@tablet u-font-size-12 u-line-height-normal lrv-u-padding-tb-00@mobile-max u-min-width-30px u-width-auto@mobile-max u-min-width-auto@mobile-max lrv-u-margin-r-050@mobile-max")
    last_week = stats[0].text.strip()
    peak = stats[1].text.strip()
    weeks = stats[2].text.strip()
    
    return {
        "this_week": rank,
        "song": song,
        "artist": artist,
        "last_week": last_week,
        "peak": peak,
        "weeks_on_chart": weeks
    }

# Test on both first and second card
print(parse_card(all_cards[0]))
print(parse_card(all_cards[1]))

{'this_week': '1', 'song': "Choosin' Texas", 'artist': 'Ella Langley', 'last_week': '2', 'peak': '1', 'weeks_on_chart': '24'}
{'this_week': '2', 'song': 'Swim', 'artist': 'BTS', 'last_week': '1', 'peak': '1', 'weeks_on_chart': '2'}


In [51]:
records = []
for card in all_cards:
    try:
        records.append(parse_card(card))
    except Exception as e:
        print(f"Skipping a card due to error: {e}")

bb_df = pd.DataFrame(records)
print(f"Got {len(bb_df)} songs")
bb_df


Got 100 songs


,this_week,song,artist,last_week,peak,weeks_on_chart
0,1,Choosin' Texas,Ella Langley,2,1,24
1,2,Swim,BTS,1,1,2
2,3,Man I Need,Olivia Dean,3,2,32
3,4,I Just Might,Bruno Mars,4,1,12
4,5,Ordinary,Alex Warren,5,1,59
...,...,...,...,...,...,...
95,96,Merry Go Round,BTS,52,52,2
96,97,Griddle,Yeat & Don Toliver,-,97,1
97,98,Silent Treatment,Freya Skye,-,98,1
98,99,Taste Back,Harry Styles,69,17,4


In [53]:
def get_billboard_chart(date):
    """
    Given a date string in YYYY-MM-DD format, 
    returns a DataFrame of the Billboard Hot 100 for that week.
    """
    url = f"https://www.billboard.com/charts/hot-100/{date}/"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    
    soup = BeautifulSoup(response.text, "html.parser")
    cards = soup.find_all("div", class_="o-chart-results-list-row-container")
    
    records = []
    for card in cards:
        try:
            records.append(parse_card(card))
        except:
            pass
    
    df = pd.DataFrame(records)
    df["chart_date"] = date  # add a column so we know which week it's from
    return df

# Test it
test_df = get_billboard_chart("2026-04-05")
print(f"Got {len(test_df)} songs")
test_df.head()


Got 100 songs


,this_week,song,artist,last_week,peak,weeks_on_chart,chart_date
0,1,Choosin' Texas,Ella Langley,2,1,24,2026-04-05
1,2,Swim,BTS,1,1,2,2026-04-05
2,3,Man I Need,Olivia Dean,3,2,32,2026-04-05
3,4,I Just Might,Bruno Mars,4,1,12,2026-04-05
4,5,Ordinary,Alex Warren,5,1,59,2026-04-05


In [55]:
from datetime import datetime, timedelta

# Generate the last 10 week dates (Billboard updates on Saturdays)
today = datetime.today()
# Go back to the most recent Saturday
days_since_saturday = (today.weekday() - 5) % 7
last_saturday = today - timedelta(days=days_since_saturday)

dates = [(last_saturday - timedelta(weeks=i)).strftime("%Y-%m-%d") for i in range(10)]
print("Fetching charts for these dates:")
for d in dates:
    print(d)
    

Fetching charts for these dates:
2026-04-11
2026-04-04
2026-03-28
2026-03-21
2026-03-14
2026-03-07
2026-02-28
2026-02-21
2026-02-14
2026-02-07


In [56]:
all_weeks = []

for date in dates:
    print(f"Fetching {date}...")
    week_df = get_billboard_chart(date)
    all_weeks.append(week_df)

# Combine all weeks into one big DataFrame
billboard_all = pd.concat(all_weeks, ignore_index=True)

print(f"\nTotal rows: {len(billboard_all)}")
billboard_all


Fetching 2026-04-11...
Fetching 2026-04-04...
Fetching 2026-03-28...
Fetching 2026-03-21...
Fetching 2026-03-14...
Fetching 2026-03-07...
Fetching 2026-02-28...
Fetching 2026-02-21...
Fetching 2026-02-14...
Fetching 2026-02-07...

Total rows: 1000


,this_week,song,artist,last_week,peak,weeks_on_chart,chart_date
0,1,Choosin' Texas,Ella Langley,2,1,24,2026-04-11
1,2,Swim,BTS,1,1,2,2026-04-11
2,3,Man I Need,Olivia Dean,3,2,32,2026-04-11
3,4,I Just Might,Bruno Mars,4,1,12,2026-04-11
4,5,Ordinary,Alex Warren,5,1,59,2026-04-11
...,...,...,...,...,...,...,...
995,96,"Girl, Get Up.",Doechii & SZA,98,57,4,2026-02-07
996,97,Mrs Magic,Strawberry Guy,-,97,1,2026-02-07
997,98,Dano,Peso Pluma & Tito Double P,-,75,3,2026-02-07
998,99,3am,Loe Shimmy & Don Toliver,-,71,10,2026-02-07
